In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Loading the training and testing data

In [ ]:
df_train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')  # Assuming you have a training set
df_test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

Checking the missing data and dropping the top 6 columns of the missing data.

In [ ]:
missing = (
    df_train.isnull().sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda x: 100 * x.missing_count / len(df_train))
    .query("missing_count > 0")          
    .sort_values("missing_pct", ascending=False)
)

print(missing.shape[0], "columns have missing values")
missing.head(20)   

# Feature Engineering (Possible)

Create "House Age" by (YrSold - YearBuilt)

In [ ]:
df_train['HouseAge'] = df_train['YrSold'] - df_train['YearBuilt']
df_test['HouseAge'] = df_test['YrSold'] - df_test['YearBuilt']

Create "Total Area" by (GrLivArea + TotRmsAbvGrd)

In [ ]:
df_train['TotalArea'] = df_train['GrLivArea'] + df_train['TotRmsAbvGrd']
df_test['TotalArea'] = df_test['GrLivArea'] + df_test['TotRmsAbvGrd']

# Finding the correlation

Top 10 correlation feations and check the skewness. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr_matrix = df_train[df_train.select_dtypes(include=['number']).columns].corr()
saleprice_corr = corr_matrix['SalePrice'].sort_values(ascending=False)

top10_features = saleprice_corr.drop('SalePrice').head(10).index.tolist()
top10_corrs = saleprice_corr[top10_features]

plt.figure(figsize=(8, 5))
sns.barplot(x=top10_corrs.index, y=top10_corrs.values, color="blue")
plt.xticks(rotation=45)
plt.ylabel("Correlation with SalePrice")
plt.title("Correlation of Top 10 Features with SalePrice")
plt.tight_layout()
plt.show()

**Check the Skewness**

> 0 is right skew and < 0 is left skew

In [ ]:
# Calculate skewness for top 10 features
skewness = df_train[top10_features].skew()

# Sort skewness by absolute value
sorted_skewness = skewness.abs().sort_values()

# Print skewness values in ascending order
print(skewness[sorted_skewness.index])

In [ ]:
# Get the top 10 features correlated with SalePrice
saleprice_corr = corr_matrix['SalePrice'].sort_values(ascending=False)
top10_features = saleprice_corr.drop('SalePrice').head(10).index.tolist()

# Create subplots for visualizing histograms
fig, axes = plt.subplots(2, 5, figsize=(15, 8))  # Adjust grid size based on number of features
axes = axes.flatten()

# Plot histograms for the top 10 features
for i, feature in enumerate(top10_features):  # Use top10_features instead of top_features
    axes[i].hist(df_train[feature], bins=30, color='skyblue', edgecolor='black')
    axes[i].set_title(f'Histogram of {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

*Log Transformation for features that have greater than 1. 3 of right skewness*****

In [ ]:
#GrLivArea
df_train['GrLivArea_log'] = np.log1p(df_train['GrLivArea'])
df_test['GrLivArea_log'] = np.log1p(df_test['GrLivArea'])

Apply log transformation to Sale Price (target data)

In [ ]:
df_train['SalePrice_log'] = np.log1p(df_train['SalePrice'])

# Stratified Sampling

we use the stratified sampling to have equal allocation of the data based on SalePrice

In [ ]:
df_train["sale_category"] = pd.cut(df_train["SalePrice"], bins=[0, 100000, 200000, 300000, 400000, np.inf], 
                                   labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

# Splitting the data

In [ ]:
X_train = df_train.drop(columns=['SalePrice', 'Id', 'sale_category', 'SalePrice_log', 'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu'])  # Dropping target, Id, and SalePrice_log column
y_train = df_train['SalePrice_log']  

X_test = df_test.drop(columns=['Id'])  

**Define the columns for categorical and numerical features**

In [ ]:
num_cols = X_train.select_dtypes(include=['number']).columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

Split the training data based on stratified sampling data

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

stratified_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, val_idx in stratified_split.split(X_train, df_train["sale_category"]):
    X_train_stratified, X_val_stratified = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_train_stratified, y_val_stratified = y_train.iloc[train_idx], y_train.iloc[val_idx]

# Buidling Data Pipeline

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Create preprocessing pipelines
cat_pipeline = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)

num_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

# Create the preprocessing pipeline that combines both numerical and categorical pipelines
preprocessing = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

fitting and transforming the data

In [ ]:
# Preprocess the training and validation data
X_train_prepared = preprocessing.fit_transform(X_train_stratified)
X_val_prepared = preprocessing.transform(X_val_stratified)

# LightGBM Model (LGBM Regressor)

Initialize the model

In [ ]:
import lightgbm as lgb

lgbm_reg = lgb.LGBMRegressor(
    n_estimators=300,    # Number of boosting iterations (trees)
    max_depth=12,        # Maximum depth of each tree
    random_state=42,     # For reproducibility
    n_jobs=-1            # Use all available CPU cores
)

10 Fold Cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(lgbm_reg, X_train_prepared, y_train_stratified, cv=5, scoring='neg_root_mean_squared_error')

In [ ]:
print(f"Cross-Validation RMSE: {np.mean(np.abs(cv_scores)):.4f} ± {np.std(np.abs(cv_scores)):.4f}")

Fitting the model on the entire training data

In [ ]:
lgbm_reg.fit(X_train_prepared, y_train_stratified)

Make prediction on the validation set and evaluate

In [ ]:
from sklearn.metrics import mean_squared_error

y_val_pred = lgbm_reg.predict(X_val_prepared)

rmse = np.sqrt(mean_squared_error(y_val_stratified, y_val_pred))

print(f"Validation RMSE: {rmse:.4f}")

**Prepare the test data in the same data pipeline**

In [ ]:
X_test_prepared = preprocessing.transform(X_test)

y_test_pred = lgbm_reg.predict(X_test_prepared)

**Submission**

Combination of all the code above

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Load training and test data
df_train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')  # Assuming you have a training set
df_test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Feature Engineering:
# 1. Create 'HouseAge' feature
df_train['HouseAge'] = df_train['YrSold'] - df_train['YearBuilt']
df_test['HouseAge'] = df_test['YrSold'] - df_test['YearBuilt']

# 2. Create 'TotalArea' feature (sum of GrLivArea and TotRmsAbvGrd)
df_train['TotalArea'] = df_train['GrLivArea'] + df_train['TotRmsAbvGrd']
df_test['TotalArea'] = df_test['GrLivArea'] + df_test['TotRmsAbvGrd']

# 3. Apply log transformation to 'GrLivArea' to handle skewness
df_train['GrLivArea_log'] = np.log1p(df_train['GrLivArea'])
df_test['GrLivArea_log'] = np.log1p(df_test['GrLivArea'])

# 4. Apply log transformation to 'SalePrice' to make it more normal
df_train['SalePrice_log'] = np.log1p(df_train['SalePrice'])

# 5. Create a new 'sale_category' column based on 'SalePrice' bins
df_train["sale_category"] = pd.cut(df_train["SalePrice"], bins=[0, 100000, 200000, 300000, 400000, np.inf], 
                                   labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

# Specify the features (X) and target (y)
X_train = df_train.drop(columns=['SalePrice', 'Id', 'sale_category', 'SalePrice_log'])  # Dropping target, Id, and SalePrice_log column
y_train = df_train['SalePrice_log']  # Using the log-transformed SalePrice as target

X_test = df_test.drop(columns=['Id'])  # Dropping 'Id' from test set

# Define column names for categorical and numerical features
num_cols = X_train.select_dtypes(include=['number']).columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

# Perform stratified sampling based on the 'sale_category' column
stratified_split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, val_idx in stratified_split.split(X_train, df_train["sale_category"]):
    X_train_stratified, X_val_stratified = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_train_stratified, y_val_stratified = y_train.iloc[train_idx], y_train.iloc[val_idx]

# Create preprocessing pipelines
cat_pipeline = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)

num_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

# Create the preprocessing pipeline that combines both numerical and categorical pipelines
preprocessing = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Preprocess the training and validation data
X_train_prepared = preprocessing.fit_transform(X_train_stratified)
X_val_prepared = preprocessing.transform(X_val_stratified)

# Initialize LightGBM model (LGBMRegressor)
lgbm_reg = lgb.LGBMRegressor(
    n_estimators=300,    # Number of boosting iterations (trees)
    max_depth=12,        # Maximum depth of each tree
    random_state=42,     # For reproducibility
    n_jobs=-1            # Use all available CPU cores
)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(lgbm_reg, X_train_prepared, y_train_stratified, cv=5, scoring='neg_root_mean_squared_error')

# Output the mean and standard deviation of the cross-validation RMSE
print(f"Cross-Validation RMSE: {np.mean(np.abs(cv_scores)):.4f} ± {np.std(np.abs(cv_scores)):.4f}")

# Fit the model on the entire training data
lgbm_reg.fit(X_train_prepared, y_train_stratified)

# Make predictions on the validation set and evaluate performance
y_val_pred = lgbm_reg.predict(X_val_prepared)
rmse = np.sqrt(mean_squared_error(y_val_stratified, y_val_pred))
r2 = r2_score(y_val_stratified, y_val_pred)
print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation R²: {r2:.4f}")

# Preprocess the test data using the same preprocessing pipeline
X_test_prepared = preprocessing.transform(X_test)

# Make predictions on the test data using the trained LightGBM model
y_test_pred = lgbm_reg.predict(X_test_prepared)

# Prepare the submission DataFrame
submission_df = pd.DataFrame({
    'Id': df_test['Id'],  # Make sure 'Id' is in df_test
    'SalePrice': np.expm1(y_test_pred)  # Inverse log transformation to get the real SalePrice
})

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission.csv', index=False)

print("LightGBM submission file created.")